# 01. Train MobileNetV3-Large Classifier

**Smart Recycling Coach** — COMP 365 Team Project.
Fine-tunes an ImageNet-pretrained MobileNetV3 on the three-class recycling dataset (`recycling`, `special_handling`, `trash`).

### Running on Google Colab:
1. Choose **Runtime > Change runtime type > T4 GPU**.
2. Run the first cell below to set up the repository and dependencies in Colab.
3. Select **Runtime > Run all** to execute the pipeline end-to-end.

In [ ]:
# Colab Environment Setup (Runs automatically if in Google Colab)
import sys, os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = "COLAB_GPU" in os.environ

if IN_COLAB:
    print("🚀 Detected Google Colab environment.")
    repo_dir = Path("/content/insert-team-name-here")
    
    # 1. Clone repository if not present, or pull latest changes
    if not repo_dir.exists():
        print("Cloning repository from GitHub...")
        !git clone https://github.com/rickeyjohnson/insert-team-name-here.git /content/insert-team-name-here
    else:
        print("Repository folder exists. Pulling latest commits...")
        !cd /content/insert-team-name-here && git pull
    
    # 2. Set current working directory to the repository root
    os.chdir(str(repo_dir))
    
    # 3. Explicitly add src to sys.path with absolute path
    src_path = str(repo_dir / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    
    # 4. Install dependencies and package
    !pip install -q imagehash
    !pip install -q -e .
    print(f"✅ Working directory: {os.getcwd()}")
    print(f"✅ Added to sys.path: {src_path}")
else:
    # Local laptop execution
    local_src = Path.cwd() / "src" if (Path.cwd() / "src").exists() else Path.cwd().parent / "src"
    src_path = str(local_src.resolve())
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    print(f"✅ Running locally. Added to sys.path: {src_path}")

import torch
import pandas as pd
import matplotlib.pyplot as plt
from recycle_coach import config, ingest, clean, splits, baseline, metrics, dataset, train, predict

print(f"PyTorch: {torch.__version__} | Device: {train.get_device()}")
print(f"Canonical Classes: {config.CLASSES} | Seed: {config.SEED}")

## 1. Verify or Prepare Dataset Splits
The committed `splits.csv` (seed 365, grouped by source photo) is the split we use. A fresh clone has no image files, so if any image in `splits.csv` is missing from `data/processed/`, we download TACO and rebuild the processed crops. The committed `splits.csv`, `clean.csv` and `images.csv` are not overwritten.

In [ ]:
import tempfile

df_splits = pd.read_csv(config.SPLITS_CSV)
def missing_images():
    return [i for i in df_splits["image_id"] if not (config.PROCESSED_DIR / f"{i}.jpg").exists()]

missing = missing_images()
if missing:
    print(f"{len(missing)} of {len(df_splits)} processed images are missing. Rebuilding crops from TACO (a few minutes)...")
    tmp = Path(tempfile.mkdtemp())
    ingest.ingest_taco_dataset(cap_per_class=450, output_csv=tmp / "images.csv")
    clean.run_cleaning(images_csv=tmp / "images.csv", output_clean_csv=tmp / "clean.csv")
    missing = missing_images()
assert not missing, f"Still missing {len(missing)} images, e.g. {missing[:5]}. Check the TACO download log above."
print(f"All {len(df_splits)} images in splits.csv are present in data/processed/.")

split_counts = pd.crosstab(df_splits["final_label"], df_splits["split"]).reindex(
    index=list(config.CLASSES), columns=["train", "val", "test"]
)
split_counts.loc["Total"] = split_counts.sum()
print(split_counts)


## 2. Evaluate Majority-Class Baseline
Compute the majority-class baseline benchmark on the validation split.

In [ ]:
import json
base_val_preds = baseline.run_baseline(split_name="val")
base_test_preds = baseline.run_baseline(split_name="test")
with open(config.METRICS_DIR / "baseline_val.json") as f:
    base_val_metrics = json.load(f)
with open(config.METRICS_DIR / "baseline_test.json") as f:
    base_test_metrics = json.load(f)

print(f"Baseline Val  Accuracy: {base_val_metrics['accuracy']:.4f} | Macro F1: {base_val_metrics['macro_f1']:.4f}")
print(f"Baseline Test Accuracy: {base_test_metrics['accuracy']:.4f} | Macro F1: {base_test_metrics['macro_f1']:.4f}")


## 3. Two-Stage MobileNetV3 Fine-Tuning
- **Stage 1:** Freeze backbone, train classifier head (lr=1e-3, 3 epochs).
- **Stage 2:** Unfreeze backbone, full fine-tuning with class-weighted cross-entropy and Cosine Annealing (lr=1e-4, patience=5).

In [ ]:
training_results = train.train_model(
    stage1_epochs=3,
    stage2_epochs=15,
    patience=5,
    batch_size=32,
    pretrained=True
)

best_epoch = training_results['best_epoch']
best_f1 = training_results['best_macro_f1']
print(f"Best Checkpoint: Epoch {best_epoch} | Val Macro F1: {best_f1:.4f}")

## 4. Plot Training & Validation Curves

In [ ]:
h = pd.DataFrame(training_results["history"])  # stage-2 epochs; also saved to results/metrics/mobilenetv3_history.csv
print(h.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(h["epoch"], h["train_loss"], marker="o", color="#2a78d6")
ax[0].set_title("Training loss (stage 2)"); ax[0].set_xlabel("Stage-2 epoch"); ax[0].grid(True, alpha=0.3)

ax[1].plot(h["epoch"], h["val_macro_f1"], marker="o", label="Val macro F1", color="#2a78d6")
ax[1].plot(h["epoch"], h["val_accuracy"], marker="s", label="Val accuracy", color="#eb6834")
ax[1].axhline(base_val_metrics["macro_f1"], ls="--", color="gray", label=f"Baseline macro F1 ({base_val_metrics['macro_f1']:.2f})")
ax[1].axvline(best_epoch, ls=":", color="black", alpha=0.5, label=f"Best epoch ({best_epoch})")
ax[1].set_ylim(0, 1)
ax[1].set_title(f"Validation (n={int((df_splits['split'] == 'val').sum())})"); ax[1].set_xlabel("Stage-2 epoch")
ax[1].legend(fontsize=9); ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "training_curves.png", dpi=150)
plt.show()


## 5. Held-Out Test Evaluation
Evaluate the trained MobileNetV3 model on the held-out test split.

In [ ]:
# Run predictions on test split
predictor = predict.load_predictor(checkpoint_path=config.CHECKPOINT_PATH)
df_test = df_splits[df_splits["split"] == "test"].reset_index(drop=True)

test_records = []
for _, row in df_test.iterrows():
    img_path = config.PROCESSED_DIR / f"{row['image_id']}.jpg"
    res = predictor.predict(img_path)
    test_records.append({
        "image_id": row["image_id"],
        "split": "test",
        "true_label": row["final_label"],
        "pred_label": res["label"],
        "confidence": res["confidence"],
        "p_recycling": res["probs"].get("recycling", 0.0),
        "p_special_handling": res["probs"].get("special_handling", 0.0),
        "p_trash": res["probs"].get("trash", 0.0),
    })

test_preds_csv = config.PREDICTIONS_DIR / "mobilenetv3_test.csv"
pd.DataFrame(test_records).to_csv(test_preds_csv, index=False)

test_eval = metrics.evaluate_predictions(
    predictions_csv=test_preds_csv,
    model_name="mobilenetv3",
    split_name="test",
    output_metrics_json=config.METRICS_DIR / "mobilenetv3_test.json",
    output_figure_png=config.FIGURES_DIR / "confusion_mobilenetv3_test.png",
    output_error_csv=config.ERRORS_DIR / "mobilenetv3_test_errors.csv",
)

print(f"Held-Out Test Accuracy: {test_eval['accuracy']:.4f}")
print(f"Held-Out Test Macro F1: {test_eval['macro_f1']:.4f}")
print(f"Baseline Test Accuracy: {base_test_metrics['accuracy']:.4f}")
print(f"Baseline Test Macro F1: {base_test_metrics['macro_f1']:.4f}")
print(f"Special handling recall: {test_eval['per_class']['special_handling']['recall']:.3f} (n={test_eval['per_class']['special_handling']['support']})")

## 6. Live Prediction Demo
Test prediction on a sample photo or your own uploaded photo.

In [ ]:
# One fixed test image per class, so the output is the same on every run
for label in config.CLASSES:
    row = df_test[df_test["final_label"] == label].iloc[0]
    result = predictor.predict(config.PROCESSED_DIR / f"{row['image_id']}.jpg")
    print(f"{row['image_id']} (true: {label})")
    print(json.dumps(result, indent=2))


## 7. Download Checkpoint and Results
Downloads `mobilenetv3_best.pt` plus a zip of `results/` (history, metrics, predictions, errors, figures) so they can be committed.

In [ ]:
import shutil
if IN_COLAB and config.CHECKPOINT_PATH.exists():
    from google.colab import files
    zip_path = shutil.make_archive("/content/cp2_results", "zip", root_dir=config.ROOT, base_dir="results")
    for path in (str(config.CHECKPOINT_PATH), zip_path):
        print(f"Downloading {Path(path).name}...")
        files.download(path)
else:
    print("Not in Colab: the checkpoint is at", config.CHECKPOINT_PATH, "and results are in", config.RESULTS_DIR)
